## Vector search

Importamos métodos definidos en el módulo 1:

In [1]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

Cargamos documentos:

In [2]:
documents = load_faq_data()
doc_texts = []

for doc in documents:
    doc_texts.append(doc['question']+ " " + doc['answer'])


Generamos los embeddings:

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
from tqdm.auto import tqdm

batch_size = 50
doc_embeddings = []

for i in tqdm(range(0, len(doc_texts), batch_size)):
    batch = doc_texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    doc_embeddings.extend(batch_vectors)

len(doc_texts)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [5]:
import numpy as np

X = np.array(doc_embeddings)

print(X.shape)

(1350, 384)


Vamos a adaptar la clase RAGBase para que emplee la búsqueda vectorial de minsearch (VectorSearch).

En primer lugar, generamos un nuevo índice:

In [6]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

Creamos una subclase de RAGBase para definir un nuevo método de búsqueda:

In [7]:
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
    
    def search(self, query, num_results=5):
        query_embedded = self.embedder.encode(query)
        filter_dict = {'course': self.course}
        return self.index.search(
            query_embedded,
            num_results=num_results,
            filter_dict = filter_dict
        ) 


rag_vector = RAGVector(embedder=model, index=vindex)

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

rag_vector.llm_client.api_key=os.getenv('GROQ_API_KEY')

In [9]:
rag_vector.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up for the course. According to the context, "You can also just start learning and submitting homework (while the form is open) without registering." However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'